### Estimating Dipole Moment of CaS

Using the pyscf and pyberny packages. Code generated by Gemini Pro 3, edited and commented by BAM.

In [1]:
import pyscf
from pyscf import dft
from pyscf.geomopt.berny_solver import optimize 
from pyscf.hessian import thermo

In [3]:
#Define CaS
mol = pyscf.M(
    atom='Ca 0.0 0.0 0.0; S 0.0 0.0 2.85',
    #basis='6-311++g(d,p)',
    basis='def2-QZVPPD',
    spin=0,
    charge=0,
    unit='Angstrom'
)

In [4]:
#Build the Unrestricted DFT object (UKS) and set the functional
mf = dft.UKS(mol)
mf.xc = 'm06-2x'

In [5]:
#Perform the Geometry Optimization
print("--- Starting Geometry Optimization (M06-2X / def2-QZVPPD) ---")
mol_eq = optimize(mf)

--- Starting Geometry Optimization (M06-2X / def2-QZVPPD) ---

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
  Ca   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   S   0.000000   0.000000   2.850000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -1075.75888673983  <S^2> = 1.7763568e-12  2S+1 = 1
--------------- UKS_Scanner gradients ---------------
         x                y                z
0 Ca    -0.0000000000     0.0000000000    -0.0552630534
1 S    -0.0000000000    -0.0000000000     0.0552235241
----------------------------------------------
cycle 1: E = -1075.75888674  dE = -1075.76  norm(grad) = 0.0781258

Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
  Ca   0.000000   0.000000   0.033035    0.000000  0.000000  0.033035
   S   0.000000   0.000000

In [6]:
print("\nOptimized Geometry Structure:")
print(mol_eq.tostring())


Optimized Geometry Structure:
Ca          0.00000000        0.00000000        0.27239537
S           0.00000000        0.00000000        2.57760463


In [7]:
#Run a final single-point calculation on the optimized structure.
# 'optimize()' returns a new Mole object (mol_eq) with updated coordinates.
print("\n--- Running Final Calculation on Optimized Geometry ---")
mf_eq = dft.UKS(mol_eq)
mf_eq.xc = 'm06-2x'
mf_eq.kernel()


--- Running Final Calculation on Optimized Geometry ---
converged SCF energy = -1075.80066089352  <S^2> = 4.9585225e-11  2S+1 = 1


-1075.8006608935175

In [8]:
#Extract and print the final dipole moment
print("\n--- Final Dipole Moment Results ---")
dipole_vector = mf_eq.dip_moment()


--- Final Dipole Moment Results ---
Dipole moment(X, Y, Z, Debye):  0.00000, -0.00000, -11.05618


In [10]:
#Compute Rotational Constants for comparison to lab work (B = 5284 from Takano:1989:563)
print("\n=== ROTATIONAL CONSTANT ===")
# PySCF stores coordinates in Bohr internally; thermo.rotation_const expects Bohr and AMU
masses = mol_eq.atom_mass_list()
coords = mol_eq.atom_coords()

# Calculate the constants in MHz
rot_constants = thermo.rotation_const(masses, coords, unit='GHz')*1000.
print(f'A: {rot_constants[0]:.2f} MHz\nB: {rot_constants[1]:.2f} MHz\nC: {rot_constants[2]:.2f} MHz')


=== ROTATIONAL CONSTANT ===
A: inf MHz
B: 5349.57 MHz
C: 5349.57 MHz


Given that these are B_e values, this is close enough to the Takano work for the dipole to be a reasonable estimate.